# Dán nhãn bình luận tiếng Việt bằng Gemini

Notebook này dùng codebook trong `guidelines/`; prompt được nhúng trực tiếp trong notebook để chạy độc lập.
LLM chỉ tiền dán nhãn; các bản ghi review và bất đồng phải được người kiểm duyệt xác nhận.

Schema nhãn: `TOXIC/NON-TOXIC` và, nếu TOXIC, `EXPLICIT/IMPLICIT`.

In [ ]:
from pathlib import Path
import hashlib
import html
import json
import re
import time

import pandas as pd
import google.generativeai as genai
from google.colab import userdata

# Tự tìm thư mục Annotation đã tổ chức.
def find_annotation_dir():
    candidates = [Path.cwd(), Path.cwd() / 'Annotation', Path.cwd().parent, Path.cwd().parent / 'Annotation', Path('/content/Annotation')]
    for candidate in candidates:
        if (candidate / 'guidelines' / 'label_definitions.md').exists():
            return candidate.resolve()
    raise FileNotFoundError('Không tìm thấy thư mục Annotation đã tổ chức.')

ANNOTATION_DIR = find_annotation_dir()
INPUT_PATH = ANNOTATION_DIR.parent / 'Preprocessing' / 'data' / 'raw.csv'
OUTPUT_PATH = ANNOTATION_DIR / 'outputs' / 'annotation_gemini.csv'
MODEL_NAME = 'gemini-2.0-flash'
PROMPT_VERSION = 'ds108-inline-v2'
MAX_ROWS = None  # Đặt số nguyên khi chạy thử; None để chạy toàn bộ batch.
START_ROW_ID = None

api_key = userdata.get('GOOGLE_API_KEY')
if not api_key:
    raise RuntimeError('Chưa cấu hình GOOGLE_API_KEY trong Colab Secrets.')
genai.configure(api_key=api_key)

generation_config = {
    'temperature': 0.0,
    'top_p': 1.0,
    'max_output_tokens': 600,
}


In [ ]:
PROMPT_TEMPLATE = """ROLE
Bạn là annotator cho bình luận tiếng Việt. Chỉ đọc Comment được cung cấp. Đọc
toàn câu, không dò từ khóa riêng lẻ, không suy diễn ngữ cảnh ngoài Comment và
không tự tạo nhãn mới.

LABELS
- TOXIC: có xúc phạm, lăng mạ, đe dọa, quấy rối, hạ nhục, phân biệt, chế giễu
  hoặc công kích gây tổn thương/làm cuộc trò chuyện tiêu cực.
- NON-TOXIC: tích cực, trung tính, hỏi đáp hoặc phản biện ý tưởng/hành vi/sự
  việc mà không công kích người hay nhóm người.
- EXPLICIT: nội dung độc hại trực tiếp, rõ ràng.
- IMPLICIT: nội dung độc hại qua mỉa mai, ám chỉ, hàm ý hoặc khuôn mẫu hóa.

DECISION RULES
1. Không có căn cứ công kích → toxicity=NON-TOXIC.
2. Có căn cứ công kích → toxicity=TOXIC.
3. Với TOXIC, trực tiếp/rõ ràng → expression_type=EXPLICIT; ẩn ý/mỉa mai/hàm
   ý → expression_type=IMPLICIT.
4. Từ tục không tự động là TOXIC; xét chức năng trong toàn câu.
5. Phê bình ý tưởng, hành vi hoặc sự việc không phải công kích con người.
6. Trích dẫn để phản bác/lên án rõ ràng → NON-TOXIC; nếu dùng để công kích →
   đánh giá theo nội dung độc hại.
7. Emoji, teencode, tiếng Việt không dấu và code-mixing phải được đọc cùng câu.
8. Nếu thiếu căn cứ hoặc có nhiều cách hiểu hợp lý → tất cả trường nhãn là null,
   needs_review=true và ghi review_reason.

OUTPUT RULES
- NON-TOXIC → expression_type=null.
- TOXIC → expression_type=EXPLICIT hoặc IMPLICIT.
- Chỉ trả về một JSON object hợp lệ, không Markdown, không giải thích ngoài JSON.

FEW-SHOT
Comment: \"Bài này hay quá 😂👍\"
Output: {\"toxicity\":\"NON-TOXIC\",\"expression_type\":null,\"evidence\":\"Lời khen, không công kích.\",\"needs_review\":false,\"review_reason\":null}

Comment: \"Mày đúng là đồ vô dụng, cút đi.\"
Output: {\"toxicity\":\"TOXIC\",\"expression_type\":\"EXPLICIT\",\"evidence\":\"Xúc phạm trực tiếp một người.\",\"needs_review\":false,\"review_reason\":null}

Comment: \"Đúng là thằng đầu đất làm gì cũng hỏng.\"
Output: {\"toxicity\":\"TOXIC\",\"expression_type\":\"IMPLICIT\",\"evidence\":\"Mỉa mai để hạ thấp người khác.\",\"needs_review\":false,\"review_reason\":null}

Comment: \"Bài này lập luận chưa thuyết phục, nhưng mình tôn trọng tác giả.\"
Output: {\"toxicity\":\"NON-TOXIC\",\"expression_type\":null,\"evidence\":\"Phản biện lập luận mà không công kích người viết.\",\"needs_review\":false,\"review_reason\":null}

Comment: \"Có người nói 'dân X toàn ngu', nhưng đó là định kiến vô căn cứ.\"
Output: {\"toxicity\":\"NON-TOXIC\",\"expression_type\":null,\"evidence\":\"Đang phản bác nội dung độc hại được trích dẫn.\",\"needs_review\":false,\"review_reason\":null}

Comment: \"Hay quá ha 😭\"
Output: {\"toxicity\":null,\"expression_type\":null,\"evidence\":\"Có thể là khen hoặc mỉa mai; thiếu ngữ cảnh.\",\"needs_review\":true,\"review_reason\":\"Không xác định được ý định từ comment đơn lẻ.\"}

INPUT
Comment: {comment}

OUTPUT JSON
{
  \"toxicity\": \"TOXIC | NON-TOXIC | null\",
  \"expression_type\": \"EXPLICIT | IMPLICIT | null\",
  \"evidence\": \"một câu ngắn dựa trên comment\",
  \"needs_review\": true,
  \"review_reason\": \"null hoặc lý do cần người kiểm duyệt\"
}"""

def clean_text(value):
    text = '' if pd.isna(value) else str(value)
    text = html.unescape(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def render_prompt(comment):
    return (PROMPT_TEMPLATE
            .replace('{comment}', clean_text(comment)))


In [ ]:
ALLOWED_TOXICITY = {'TOXIC', 'NON-TOXIC'}

def validate_annotation(annotation):
    required = {'toxicity', 'expression_type', 'evidence',
                'needs_review', 'review_reason'}
    if not required.issubset(annotation):
        raise ValueError(f'Thiếu trường: {sorted(required - set(annotation))}')

    toxicity = annotation['toxicity']
    if toxicity not in ALLOWED_TOXICITY and not (toxicity is None and annotation['needs_review']):
        raise ValueError(f'toxicity không hợp lệ: {toxicity}')

    if toxicity == 'NON-TOXIC':
        if annotation['expression_type'] is not None:
            raise ValueError('NON-TOXIC phải có expression_type là null.')
    elif toxicity == 'TOXIC':
        if annotation['expression_type'] not in {'EXPLICIT', 'IMPLICIT'}:
            raise ValueError('TOXIC phải có EXPLICIT hoặc IMPLICIT.')
    else:
        if annotation['expression_type'] is not None:
            raise ValueError('Kết quả review phải có nhãn phụ là null.')
        if not annotation['needs_review']:
            raise ValueError('Kết quả null phải được đánh dấu needs_review=true.')

    if not isinstance(annotation['needs_review'], bool):
        raise ValueError('needs_review phải là boolean.')
    return annotation

def parse_json_response(text):
    cleaned = text.strip()
    cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', cleaned, flags=re.IGNORECASE)
    match = re.search(r'\{.*\}', cleaned, flags=re.DOTALL)
    if not match:
        raise ValueError('LLM không trả về JSON object.')
    return validate_annotation(json.loads(match.group(0)))


In [ ]:
model = genai.GenerativeModel(
    model_name=MODEL_NAME,
    generation_config=generation_config,
)
REQUEST_INTERVAL = 4.1  # xấp xỉ 15 request/phút
last_request_time = 0.0

def annotate(comment):
    global last_request_time
    elapsed = time.time() - last_request_time
    if elapsed < REQUEST_INTERVAL:
        time.sleep(REQUEST_INTERVAL - elapsed)

    response = model.generate_content(render_prompt(comment))
    last_request_time = time.time()
    raw_response = response.text.strip()
    return parse_json_response(raw_response), raw_response


In [ ]:
df = pd.read_csv(INPUT_PATH)
required_columns = {'comment'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Thiếu cột bắt buộc: {sorted(missing_columns)}')
if 'row_id' not in df.columns:
    df.insert(0, 'row_id', range(len(df)))

batch = df.copy()
if START_ROW_ID is not None:
    batch = batch[batch['row_id'] >= START_ROW_ID]
if MAX_ROWS is not None:
    batch = batch.head(MAX_ROWS)

records = []
for _, row in batch.iterrows():
    comment = clean_text(row.get('comment', ''))
    input_hash = hashlib.sha256(comment.encode('utf-8')).hexdigest()
    try:
        annotation, raw_response = annotate(comment)
    except Exception as exc:
        annotation = {
            'toxicity': None,
            'expression_type': None,
            'evidence': '',
            'needs_review': True,
            'review_reason': f'LLM_OR_PARSE_ERROR: {type(exc).__name__}',
        }
        raw_response = str(exc)

    records.append({
        'row_id': row['row_id'],
        'platform': row.get('platform', ''),
        'comment': comment,
        'input_hash': input_hash,
        **annotation,
        'raw_response': raw_response,
        'model': MODEL_NAME,
        'prompt_version': PROMPT_VERSION,
    })

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(records).to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'Đã lưu {len(records)} bản ghi vào {OUTPUT_PATH}')


In [ ]:
result = pd.read_csv(OUTPUT_PATH)
print(result['toxicity'].value_counts(dropna=False))
print('needs_review:', result['needs_review'].value_counts(dropna=False).to_dict())

# Các trường hợp review phải được người kiểm duyệt xử lý trước khi tạo gold label.
review_queue = result[result['needs_review'].astype(str).str.lower() == 'true']
print('review queue:', len(review_queue))
